# Clip a video around a lick using `video_alignment`

`video_alignment` converts event times from other data streams into seconds
within the recorded video, so you can clip around them. Three clocks are used:

| clock | zero point |
|---|---|
| `behavior_time` | absolute harp / reference time (video CSV `Behav_Time`, NWB events) |
| `video_time` | first video frame = 0.0 (what `ffmpeg -ss` expects) |
| `session_time` | first go cue = 0.0 |

Here we load a real session, find the **first lick after the first go cue** from
the NWB events table, convert it to `video_time`, and clip a 100 ms window
centered on the lick.

In [ ]:
import glob

from aind_dynamic_foraging_data_utils import nwb_utils
from aind_dynamic_foraging_behavior_video_analysis import video_alignment as va
from aind_dynamic_foraging_behavior_video_analysis.kinematics.video_clip_utils import (
    extract_clips_ffmpeg_after_reencode,
)

session = "behavior_791691_2025-06-27_13-54-27"
behavior_videos_path = f"/root/capsule/data/{session}/behavior-videos"
video_csv_path = f"{behavior_videos_path}/bottom_camera.csv"
video_path = f"{behavior_videos_path}/bottom_camera.mp4"

## 1. Load the NWB and its events table

Loaded directly with `aind_dynamic_foraging_data_utils` (no dependency on the
kinematics pipeline). `create_df_events` returns a tidy events dataframe whose
`timestamps` are in `session_time` (first go cue = 0).

In [ ]:
# Locate the NWB for this session (adjust the pattern if your NWB lives elsewhere)
nwb_path = glob.glob("/root/capsule/data/**/*791691*.nwb", recursive=True)[0]
print("Using NWB:", nwb_path)

nwb = nwb_utils.load_nwb_from_filename(nwb_path)
df_events = nwb_utils.create_df_events(nwb)

first_go_cue_time = float(nwb.trials["goCue_start_time"][0])  # behavior_time of the first go cue
df_events.head()

## 2. First lick following the first go cue

Licks are `left_lick_time` / `right_lick_time` events. Since `timestamps` are
session time (go cue = 0), the first lick after the go cue is the smallest
positive timestamp.

In [ ]:
licks = df_events[df_events["event"].isin(["left_lick_time", "right_lick_time"])]
first_lick_session_time = licks.loc[licks["timestamps"] > 0, "timestamps"].min()
print(f"first lick at {first_lick_session_time:.4f} s after the first go cue")

## 3. Convert the lick time to video_time

`offset` is the session↔video shift, read straight from the behavior video CSV
and the first go-cue time.

In [ ]:
offset = va.compute_video_session_offset(video_csv_path, first_go_cue_time)
lick_video_time = va.session_time_to_video_time(first_lick_session_time, offset)
print(f"lick is at {lick_video_time:.4f} s into the video")

## 4. Clip a 100 ms window centered on the lick

Start half the clip length before the lick so the lick lands in the middle of
the clip.

> Note: `extract_clips_ffmpeg_after_reencode` uses stream copy (`-c copy`), which
> cuts at keyframes; for exactly frame-centered output, re-encode instead.

In [ ]:
clip_length = 0.1  # 100 ms

extract_clips_ffmpeg_after_reencode(
    video_path,
    timestamps=[lick_video_time - clip_length / 2],
    clip_length=clip_length,
    output_dir="/root/capsule/scratch/lick_clips",
    filename_stems=["first_lick"],
)